## Импорты

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

In [ ]:
!pip install opencv-contrib-python==4.10.0.84

In [ ]:
import os
import glob
import time
import cv2
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
import itertools

### Получение изображений

In [ ]:
drive.mount('/content/drive')
print("Google Drive успешно смонтирован.")

In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/PLANKTON"

if os.path.exists(IMAGE_DIR):
    print(f"Папка найдена: {IMAGE_DIR}")
    print("\nСодержимое папки (первые 20 элементов):")
    for item in sorted(os.listdir(IMAGE_DIR))[:20]:
        print(f"  - {item}")
else:
    print(f" Папка не найдена: {IMAGE_DIR}")

In [ ]:
extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG",
              "*.bmp", "*.BMP", "*.tif", "*.tiff", "*.TIF", "*.TIFF")

image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))

image_paths = sorted(set(image_paths))

print(f"Найдено изображений: {len(image_paths)}\n")
for i, p in enumerate(image_paths, 1):
    print(f"  {i}. {os.path.relpath(p, IMAGE_DIR)}")

In [ ]:
for i, path in enumerate(image_paths, 1):
    img = cv2.imread(path)
    if img is not None:
        h, w = img.shape[:2]
        c = img.shape[2] if img.ndim == 3 else 1
        size_kb = os.path.getsize(path) / 1024
        print(f"{i}. {os.path.relpath(path, IMAGE_DIR)}")
        print(f"   Размер: {w} × {h} px | каналов: {c} | файл: {size_kb:.1f} КБ")
    else:
        print(f"{i}. {os.path.relpath(path, IMAGE_DIR)} — не удалось прочитать")

In [ ]:
records = []
for i, path in enumerate(image_paths, 1):
    img = cv2.imread(path)
    if img is not None:
        h, w = img.shape[:2]
        c = img.shape[2] if img.ndim == 3 else 1
        records.append({
            "id": i,
            "filename": os.path.basename(path),
            "path": path,
            "width": w,
            "height": h,
            "channels": c,
            "filesize_kb": round(os.path.getsize(path) / 1024, 1),
        })
    else:
        records.append({
            "id": i, "filename": os.path.basename(path), "path": path,
            "width": None, "height": None, "channels": None,
            "filesize_kb": round(os.path.getsize(path) / 1024, 1),
        })

df = pd.DataFrame(records)
print("DataFrame с метаданными изображений:")
display(df)

print("\nУникальные разрешения:")
print(df.groupby(["width", "height"]).size().reset_index(name="count"))

### Загрузка изображений + превью + статистика

In [ ]:
images_bgr = []
images_gray = []
images_rgb = []

for path in tqdm(image_paths, desc="Загрузка"):
    bgr = cv2.imread(path)
    if bgr is None:
        print(f" Не удалось прочитать: {path}")
        images_bgr.append(None); images_gray.append(None); images_rgb.append(None)
        continue
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    images_bgr.append(bgr)
    images_gray.append(gray)
    images_rgb.append(rgb)

print(f"\n Загружено изображений: {len(images_bgr)}")
print(f"   Из них валидных: {sum(1 for x in images_bgr if x is not None)}")

In [ ]:
n = len(images_rgb)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten() if n > 1 else [axes]

for i, (rgb, path) in enumerate(zip(images_rgb, image_paths)):
    ax = axes[i]
    if rgb is not None:
        ax.imshow(rgb)
        h, w = rgb.shape[:2]
        ax.set_title(f"{i+1}. {os.path.basename(path)}\n{w}×{h}", fontsize=10)
    else:
        ax.text(0.5, 0.5, "Ошибка загрузки", ha="center", va="center")
    ax.axis("off")

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle("Bosmina longirostris — 6 изображений", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
stats = []
for i, (gray, bgr, path) in enumerate(zip(images_gray, images_bgr, image_paths), 1):
    if gray is None:
        continue
    h, w = gray.shape
    mean_brightness = float(np.mean(gray))
    std_contrast = float(np.std(gray))
    pct_overexposed = float(np.mean(gray > 250) * 100)
    pct_underexposed = float(np.mean(gray < 5) * 100)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    stats.append({
        "id": i,
        "filename": os.path.basename(path),
        "width": w,
        "height": h,
        "channels": bgr.shape[2] if bgr.ndim == 3 else 1,
        "megapixels": round(w * h / 1e6, 2),
        "mean_brightness": round(mean_brightness, 1),
        "std_contrast": round(std_contrast, 1),
        "pct_overexposed": round(pct_overexposed, 2),
        "pct_underexposed": round(pct_underexposed, 2),
        "sharpness_var_lap": round(sharpness, 1),
    })

stats_df = pd.DataFrame(stats)
print(" Статистика по изображениям:")
display(stats_df)

print("\n Сводка:")
print(f"  Диапазон разрешений: {stats_df['width'].min()}×{stats_df['height'].min()} — "
      f"{stats_df['width'].max()}×{stats_df['height'].max()}")
print(f"  Мегапиксели: от {stats_df['megapixels'].min()} до {stats_df['megapixels'].max()} Мп")
print(f"  Средняя яркость: от {stats_df['mean_brightness'].min()} до {stats_df['mean_brightness'].max()}")
print(f"  Контраст (std): от {stats_df['std_contrast'].min()} до {stats_df['std_contrast'].max()}")
print(f"  Резкость (var Laplacian): от {stats_df['sharpness_var_lap'].min()} до {stats_df['sharpness_var_lap'].max()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for i, (gray, path) in enumerate(zip(images_gray, image_paths)):
    ax = axes[i]
    if gray is not None:
        ax.hist(gray.ravel(), bins=64, range=(0, 255), color="steelblue", alpha=0.8)
        ax.set_title(f"{i+1}. mean={np.mean(gray):.0f}, std={np.std(gray):.0f}", fontsize=10)
        ax.set_xlabel("Яркость")
        ax.set_ylabel("Кол-во пикселей")
        ax.grid(alpha=0.3)

plt.suptitle("Гистограммы яркости (grayscale)", fontsize=13)
plt.tight_layout()
plt.show()

## Препроцессинг: ресайз + CLAHE

In [ ]:
MAX_SIDE = 1280

def resize_keep_aspect(gray, max_side=MAX_SIDE):
    h, w = gray.shape[:2]
    s = max_side / max(h, w)
    if s >= 1.0:
        return gray, 1.0
    return cv2.resize(gray, (int(round(w*s)), int(round(h*s))),
                      interpolation=cv2.INTER_AREA), s

def upscale_to_maxside(gray, max_side=MAX_SIDE):
    """Для изображения №6 — увеличиваем до 1280, чтобы SIFT нашёл точки."""
    h, w = gray.shape[:2]
    s = max_side / max(h, w)
    if s <= 1.0:
        return gray, 1.0
    return cv2.resize(gray, (int(round(w*s)), int(round(h*s))),
                      interpolation=cv2.INTER_CUBIC), s

def apply_clahe(gray, clip, tile):
    return cv2.createCLAHE(clipLimit=clip, tileGridSize=tile).apply(gray)

gray_norm = []
scale_records = []
for i, g in enumerate(images_gray, 1):
    if max(g.shape) < MAX_SIDE:
        out, s = upscale_to_maxside(g)
        direction = "↑ upscale"
    else:
        out, s = resize_keep_aspect(g)
        direction = "↓ downscale"
    gray_norm.append(out)
    scale_records.append({"id": i, "orig": f"{g.shape[1]}×{g.shape[0]}",
                          "new": f"{out.shape[1]}×{out.shape[0]}",
                          "scale": round(s, 3), "dir": direction})

gray_raw = gray_norm
gray_clahe_mid  = [apply_clahe(g, 2.0, (8, 8)) for g in gray_norm]
gray_clahe_hard = [cv2.bilateralFilter(apply_clahe(g, 4.0, (16, 16)),
                                       d=5, sigmaColor=50, sigmaSpace=50)
                   for g in gray_norm]

print(pd.DataFrame(scale_records).to_string(index=False))

fig, axes = plt.subplots(3, 6, figsize=(22, 10))
for k in range(6):
    for row, (imgs, title) in enumerate([
        (gray_raw,        "raw"),
        (gray_clahe_mid,  "CLAHE clip=2.0"),
        (gray_clahe_hard, "CLAHE clip=4.0 + bilateral")]):
        axes[row, k].imshow(imgs[k], cmap="gray", vmin=0, vmax=255)
        axes[row, k].set_title(f"{k+1}. {title}", fontsize=9)
        axes[row, k].axis("off")
plt.suptitle("Три версии препроцессинга", fontsize=13)
plt.tight_layout()
plt.show()

### Детекция SIFT — базовые параметры

Параметры SIFT по умолчанию:

nfeatures = 0 — без ограничения, детекция всех точек;

nOctaveLayers = 3;

contrastThreshold = 0.04;

edgeThreshold = 10;

sigma = 1.6.

In [ ]:
sift = cv2.SIFT_create()

sift_soft = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=15)

def detect(imgs):
    kps, descs = [], []
    for i, im in enumerate(imgs):
        s = sift_soft if i == 5 else sift
        kp, d = s.detectAndCompute(im, None)
        kps.append(kp); descs.append(d)
    return kps, descs

kps_raw,  descs_raw  = detect(gray_raw)
kps_mid,  descs_mid  = detect(gray_clahe_mid)
kps_hard, descs_hard = detect(gray_clahe_hard)

summary = pd.DataFrame({
    "id": range(1, 7),
    "raw":        [len(k) for k in kps_raw],
    "clahe_mid":  [len(k) for k in kps_mid],
    "clahe_hard": [len(k) for k in kps_hard],
})
print("Число ключевых точек по версиям препроцессинга:")
print(summary.to_string(index=False))

### Визуализация SIFT-точек по версиям препроцессинга

In [ ]:
def draw_kp_grid(images_list, kps_list, row_labels, suptitle):
    n_rows = len(images_list)
    n_cols = len(images_list[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.2 * n_rows))

    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for r in range(n_rows):
        for c in range(n_cols):
            ax = axes[r, c]
            vis = cv2.drawKeypoints(
                images_list[r][c], kps_list[r][c], None,
                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
                color=(0, 255, 0)
            )
            ax.imshow(vis, cmap="gray", vmin=0, vmax=255)
            n_kp = len(kps_list[r][c])
            h, w = images_list[r][c].shape
            ax.set_title(f"{c+1}. {row_labels[r]}\n{n_kp} точек | {w}×{h}",
                         fontsize=9)
            ax.axis("off")

    plt.suptitle(suptitle, fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()


draw_kp_grid(
    images_list  = [gray_raw,        gray_clahe_mid,  gray_clahe_hard],
    kps_list     = [kps_raw,         kps_mid,         kps_hard],
    row_labels   = ["raw",           "CLAHE mid",     "CLAHE hard"],
    suptitle     = "SIFT ключевые точки: raw vs CLAHE mid vs CLAHE hard "
                   "(для №6 — мягкий SIFT contrastThreshold=0.005)"
)

Крупный план одного изображения

In [ ]:
IDX = 1

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
versions = [("raw",        gray_raw,        kps_raw),
            ("CLAHE mid",  gray_clahe_mid,  kps_mid),
            ("CLAHE hard", gray_clahe_hard, kps_hard)]

for ax, (name, imgs, kps) in zip(axes, versions):
    vis = cv2.drawKeypoints(
        imgs[IDX], kps[IDX], None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
        color=(0, 255, 0)
    )
    ax.imshow(vis, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"№{IDX+1} — {name}  |  {len(kps[IDX])} точек", fontsize=13)
    ax.axis("off")

plt.suptitle(f"Крупный план изображения №{IDX+1}: где SIFT находит точки",
             fontsize=14)
plt.tight_layout()
plt.show()

### Статистика размеров ключевых точек

In [ ]:
def kp_size_stats(kps, label):
    rows = []
    for i, kp_list in enumerate(kps, 1):
        if len(kp_list) == 0:
            rows.append({"id": i, "label": label, "n": 0,
                         "median_size": 0, "p10": 0, "p90": 0})
            continue
        sizes = np.array([k.size for k in kp_list])
        rows.append({
            "id": i,
            "label": label,
            "n": len(sizes),
            "median_size": round(float(np.median(sizes)), 1),
            "p10": round(float(np.percentile(sizes, 10)), 1),
            "p90": round(float(np.percentile(sizes, 90)), 1),
        })
    return pd.DataFrame(rows)

stats_df = pd.concat([
    kp_size_stats(kps_raw,  "raw"),
    kp_size_stats(kps_mid,  "CLAHE mid"),
    kp_size_stats(kps_hard, "CLAHE hard"),
], ignore_index=True)

print("Статистика размеров ключевых точек (в пикселях):")
print(stats_df.to_string(index=False))

### Финальная версия препроцессинга для matching

Логика: CLAHE mid для №1–5, raw для №6; для №6 — мягкий SIFT.

In [ ]:
imgs_final = []
for k in range(6):
    if k == 5:
        imgs_final.append(gray_raw[k])
    else:
        imgs_final.append(gray_clahe_mid[k])

kps_final, descs_final = [], []
for k, im in enumerate(imgs_final):
    s = sift_soft if k == 5 else sift
    kp, d = s.detectAndCompute(im, None)
    kps_final.append(kp)
    descs_final.append(d)

print("Финальные числа ключевых точек:")
for k in range(6):
    src = "raw + soft SIFT" if k == 5 else "CLAHE mid + SIFT"
    print(f"  №{k+1}: {len(kps_final[k]):>6} точек  ({src})")

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for k in range(6):
    vis = cv2.drawKeypoints(imgs_final[k], kps_final[k], None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
                            color=(0, 255, 0))
    axes[k].imshow(vis, cmap="gray", vmin=0, vmax=255)
    axes[k].set_title(f"{k+1}. {len(kps_final[k])} точек", fontsize=10)
    axes[k].axis("off")
plt.suptitle("Финальная версия: CLAHE mid для №1–5, raw+soft SIFT для №6",
             fontsize=13)
plt.tight_layout()
plt.show()

### Дополнительная проверка изображения №6

In [ ]:
img6 = images_gray[5]

h, w = img6.shape
scale_up = 1280 / max(h, w)
img6_up = cv2.resize(
    img6, (int(round(w * scale_up)), int(round(h * scale_up))),
    interpolation=cv2.INTER_CUBIC
)

sift_low = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=15)
kp6_a, d6_a = sift_low.detectAndCompute(img6_up, None)

img6_denoised = cv2.bilateralFilter(img6_up, d=9, sigmaColor=75, sigmaSpace=75)
kp6_c, d6_c = sift_low.detectAndCompute(img6_denoised, None)

print(f"Изображение №6:")
print(f"  База (raw, SIFT default):           {len(kps_final[5])} точек")
print(f"  A) Upscale + SIFT low-contrast:     {len(kp6_a)} точек")
print(f"  B) Upscale + denoise + SIFT low:    {len(kp6_c)} точек")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, kp, base, title in zip(
        axes,
        [kps_final[5], kp6_a, kp6_c],
        [imgs_final[5], img6_up, img6_denoised],
        ["Raw + SIFT default (в пайплайне)",
         "Upscale + SIFT low-contrast",
         "Upscale + denoise + SIFT low"]):
    vis = cv2.drawKeypoints(base, kp, None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
                            color=(0, 255, 0))
    ax.imshow(vis, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{title}  |  {len(kp)} точек", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Быстрая проверка матчинга для №6

In [ ]:
img6_in_pipeline = imgs_final[5]
kp6_in_pipeline  = kps_final[5]

print(f"Текущее состояние №6 в пайплайне:")
print(f"  Размер:              {img6_in_pipeline.shape[1]}×{img6_in_pipeline.shape[0]}")
print(f"  Точек в kps_final[5]: {len(kp6_in_pipeline)}")
print(f"  Ожидаю 5298 (если upscale + soft SIFT), так как это не первый мой запуск файла")

sift_soft_local = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=15)
kp6_cur, d6_cur = sift_soft_local.detectAndCompute(img6_in_pipeline, None)

def quick_inliers(desc_a, kp_a, desc_b, kp_b, ratio=0.75, thresh=3.0):
    """Возвращает (число raw-матчей после ratio test, число inliers после RANSAC)."""
    bf = cv2.BFMatcher(cv2.NORM_L2)
    raw = bf.knnMatch(desc_a, desc_b, k=2)
    good = [m for m, n in raw if m.distance < ratio * n.distance]
    if len(good) < 8:
        return len(good), 0
    src = np.float32([kp_a[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp_b[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, thresh,
                                  maxIters=5000, confidence=0.999)
    return len(good), int(mask.sum()) if mask is not None else 0

print(f"\n{'Вариант №6':<32} {'точек':<8} {'vs №1 (raw/inl)':<18} {'vs №3 (raw/inl)':<18} {'Σ inl':<8}")
print("-" * 90)
for name, im, kp, d in [
    ("upscale + soft SIFT (текущий)", img6_in_pipeline, kp6_cur, d6_cur)
]:

    r1, i1 = quick_inliers(d, kp, descs_final[0], kps_final[0])
    r3, i3 = quick_inliers(d, kp, descs_final[2], kps_final[2])
    print(f"{name:<32} {len(kp):<8} {f'{r1}/{i1}':<18} {f'{r3}/{i3}':<18} {i1+i3:<8}")

## Сегментация организма

In [ ]:
def segment_organism(gray):
    """Возвращает бинарную маску организма (255 = организм, 0 = фон)."""
    med = float(np.median(gray))
    diff = np.abs(gray.astype(np.float32) - med)
    diff_n = np.clip(diff / (diff.max() + 1e-6) * 255, 0, 255).astype(np.uint8)

    _, mask = cv2.threshold(diff_n, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open,  iterations=1)

    n_lab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_lab > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest = 1 + int(np.argmax(areas))
        mask = (labels == largest).astype(np.uint8) * 255

    k_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_fill, iterations=3)
    return mask

masks_final = [segment_organism(im) for im in imgs_final]

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for k in range(6):
    vis = cv2.cvtColor(imgs_final[k], cv2.COLOR_GRAY2BGR)
    cnts, _ = cv2.findContours(masks_final[k], cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cnts, -1, (0, 255, 0), 3)
    coverage = (masks_final[k] > 0).mean() * 100
    axes[k].imshow(vis[..., ::-1])
    axes[k].set_title(f"{k+1}. cover={coverage:.1f}%", fontsize=10)
    axes[k].axis("off")
plt.suptitle("Сегментация: зелёный контур = маска организма", fontsize=13)
plt.tight_layout()
plt.show()

print("\nПокрытие маски (доля кадра):")
for k in range(6):
    print(f"  №{k+1}: {(masks_final[k] > 0).mean() * 100:.1f}%")

### Применение маски к SIFT

In [ ]:
kps_masked, descs_masked = [], []
for k, (im, mask) in enumerate(zip(imgs_final, masks_final)):
    s = sift_soft if k == 5 else sift
    kp, d = s.detectAndCompute(im, mask)
    kps_masked.append(kp)
    descs_masked.append(d)

print("Точки до и после маскирования:")
print(f"{'ID':<3} {'без маски':<12} {'с маской':<12} {'отсечено %':<10}")
print("-" * 40)
for k in range(6):
    before = len(kps_final[k])
    after  = len(kps_masked[k])
    pct = (1 - after / before) * 100 if before > 0 else 0
    print(f"{k+1:<3} {before:<12} {after:<12} {pct:<10.1f}")

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for k in range(6):
    vis = cv2.drawKeypoints(imgs_final[k], kps_masked[k], None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
                            color=(0, 255, 0))
    axes[k].imshow(vis, cmap="gray", vmin=0, vmax=255)
    axes[k].set_title(f"{k+1}. {len(kps_masked[k])} точек", fontsize=10)
    axes[k].axis("off")
plt.suptitle("SIFT только внутри маски организма", fontsize=13)
plt.tight_layout()
plt.show()

## Базовый матчинг без геометрической фильтрации

In [ ]:
def match_features_knn(desc_ref, desc_target, ratio=0.75):
    if desc_ref is None or desc_target is None:
        return []

    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except cv2.error as e:
        print(f"Ошибка матчинга: {e}")
        return []

    good_matches = []
    for m, n in matches:
        if m.distance < ratio * n.distance:
            good_matches.append(m)

    return good_matches

REF_IDX = 0

match_results = {}

print(f"Сопоставление эталона (№{REF_IDX+1}) с остальными изображениями:")
print("-" * 60)

for target_idx in range(1, 6):
    desc_ref = descs_final[REF_IDX]
    desc_target = descs_final[target_idx]

    if desc_ref is None or desc_target is None:
        print(f"№{REF_IDX+1} vs №{target_idx+1}: дескрипторы отсутствуют")
        continue

    good_matches = match_features_knn(desc_ref, desc_target, ratio=0.75)
    match_results[target_idx] = good_matches

    print(f"№{REF_IDX+1} vs №{target_idx+1}: {len(good_matches)} совпадений (до RANSAC)")

def visualize_raw_matches(img1, kp1, img2, kp2, matches, title, max_draw=50):
    if not matches:
        print(f"Нет совпадений для визуализации: {title}")
        return

    matches_sorted = sorted(matches, key=lambda x: x.distance)
    vis_matches = matches_sorted[:max_draw]

    draw_params = dict(
        matchColor=(0, 255, 0),
        singlePointColor=(255, 0, 0),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    vis_img = cv2.drawMatches(
        img1, kp1, img2, kp2, vis_matches, None, **draw_params
    )

    plt.figure(figsize=(18, 8))
    plt.imshow(vis_img, cmap='gray')
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("\n" + "="*60)
print("Визуализация 'сырых' совпадений (до RANSAC):")
print("="*60)

if 1 in match_results:
    visualize_raw_matches(
        imgs_final[0], kps_final[0],
        imgs_final[1], kps_final[1],
        match_results[1],
        f"№1 vs №2: {len(match_results[1])} совпадений (показаны 50 лучших)"
    )

if 3 in match_results:
    visualize_raw_matches(
        imgs_final[0], kps_final[0],
        imgs_final[3], kps_final[3],
        match_results[3],
        f"№1 vs №4: {len(match_results[3])} совпадений (показаны 50 лучших)"
    )

if 5 in match_results:
    visualize_raw_matches(
        imgs_final[0], kps_final[0],
        imgs_final[5], kps_final[5],
        match_results[5],
        f"№1 vs №6: {len(match_results[5])} совпадений (показаны 50 лучших)"
    )

## Эксперимент с детекторами (SIFT tuned vs AKAZE)

In [ ]:
def extract_features(images, detector, detector_name):
    kps_list, descs_list = [], []
    for i, im in enumerate(images):
        if im is None:
            kps_list.append([])
            descs_list.append(None)
            continue
        kp, desc = detector.detectAndCompute(im, None)
        kps_list.append(kp)
        descs_list.append(desc)
    print(f"  [{detector_name}] Извлечено: " +
          ", ".join([f"№{i+1}: {len(kps_list[i])}" for i in range(len(images))]))
    return kps_list, descs_list

def match_and_visualize(img1, kp1, img2, kp2, desc1, desc2,
                        norm_type, title, ratio=0.75, max_draw=40):
    if desc1 is None or desc2 is None:
        print(f"  {title}: нет дескрипторов")
        return 0

    bf = cv2.BFMatcher(norm_type, crossCheck=False)
    try:
        matches = bf.knnMatch(desc1, desc2, k=2)
    except cv2.error:
        return 0

    good = [m for m, n in matches if m.distance < ratio * n.distance]
    good = sorted(good, key=lambda x: x.distance)
    vis = good[:max_draw]

    draw_params = dict(
        matchColor=(0, 255, 0),
        singlePointColor=(255, 0, 0),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    vis_img = cv2.drawMatches(img1, kp1, img2, kp2, vis, None, **draw_params)

    plt.figure(figsize=(16, 7))
    plt.imshow(vis_img, cmap='gray')
    plt.title(f"{title} | Всего после ratio: {len(good)}", fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return len(good)

sift_default = cv2.SIFT_create()

sift_tuned = cv2.SIFT_create(
    contrastThreshold=0.01,
    edgeThreshold=20,
    nOctaveLayers=5
)

akaze = cv2.AKAZE_create(
    threshold=0.001,
    nOctaves=4,
    nOctaveLayers=4
)

ref_idx, target_idx = 0, 2

print("="*60)
print(f"СРАВНЕНИЕ ДЕТЕКТОРОВ НА ПАРЕ №{ref_idx+1} vs №{target_idx+1}")
print("="*60)

print("\n1. SIFT (по умолчанию):")
kp_sift_d, desc_sift_d = extract_features(imgs_final, sift_default, "SIFT default")
n1 = match_and_visualize(
    imgs_final[ref_idx], kp_sift_d[ref_idx],
    imgs_final[target_idx], kp_sift_d[target_idx],
    desc_sift_d[ref_idx], desc_sift_d[target_idx],
    cv2.NORM_L2, "SIFT default"
)

print("\n2. SIFT (настроенный):")
kp_sift_t, desc_sift_t = extract_features(imgs_final, sift_tuned, "SIFT tuned")
n2 = match_and_visualize(
    imgs_final[ref_idx], kp_sift_t[ref_idx],
    imgs_final[target_idx], kp_sift_t[target_idx],
    desc_sift_t[ref_idx], desc_sift_t[target_idx],
    cv2.NORM_L2, "SIFT tuned (contrast=0.01, edge=20)"
)

print("\n3. AKAZE:")
kp_akaze, desc_akaze = extract_features(imgs_final, akaze, "AKAZE")
n3 = match_and_visualize(
    imgs_final[ref_idx], kp_akaze[ref_idx],
    imgs_final[target_idx], kp_akaze[target_idx],
    desc_akaze[ref_idx], desc_akaze[target_idx],
    cv2.NORM_HAMMING, "AKAZE (threshold=0.001)"
)

print("\n" + "="*60)
print("ИТОГИ:")
print(f"SIFT default: {n1} совпадений")
print(f"SIFT tuned:   {n2} совпадений")
print(f"AKAZE:        {n3} совпадений")
print("="*60)

## Попытка с AKAZE + RANSAC + последовательное отслеживание

In [ ]:
from collections import defaultdict, Counter

try:
    akaze = cv2.AKAZE.create(
        threshold=0.00001,
        descriptor_size=0,
        descriptor_channels=3,
        nOctaves=4,
        nOctaveLayers=4,
        diffusivity=cv2.KAZE_DIFF_PM_G2
    )
    print("AKAZE успешно инициализирован через cv2.AKAZE.create()")
except AttributeError:
    akaze = cv2.AKAZE_create(
        threshold=0.0001,
        descriptor_size=0,
        descriptor_channels=3,
        nOctaves=4,
        nOctaveLayers=4,
        diffusivity=cv2.KAZE_DIFF_PM_G2
    )
    print("AKAZE инициализирован через устаревший cv2.AKAZE_create()")

kps_list, descs_list = [], []
imgs_processed = []

for i, im in enumerate(imgs_final):
    if im is None:
        kps_list.append([])
        descs_list.append(None)
        imgs_processed.append(None)
        continue

    if i == 5:
        h, w = im.shape
        scale = 1280 / max(h, w)
        if scale > 1.0:
            im = cv2.resize(im, (int(round(w*scale)), int(round(h*scale))),
                            interpolation=cv2.INTER_CUBIC)

    kp, desc = akaze.detectAndCompute(im, None)
    kps_list.append(kp)
    descs_list.append(desc)
    imgs_processed.append(im)
    print(f"№{i+1}: {len(kp)} точек AKAZE")

REF_IDX = 0
tracks = defaultdict(dict)
track_dists = defaultdict(dict)

print("\nСопоставление эталона с остальными (AKAZE + RANSAC):")
for target_idx in range(1, 6):
    desc_ref = descs_list[REF_IDX]
    desc_target = descs_list[target_idx]
    if desc_ref is None or desc_target is None:
        continue

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except cv2.error:
        continue

    good = [m for m, n in matches if m.distance < 0.8 * n.distance]
    if len(good) < 8:
        print(f"  №1 vs №{target_idx+1}: {len(good)} совпадений (мало для RANSAC)")
        continue

    src_pts = np.float32([kps_list[REF_IDX][m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kps_list[target_idx][m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    F, mask = cv2.findFundamentalMat(
        src_pts, dst_pts,
        method=cv2.FM_RANSAC,
        ransacReprojThreshold=7.0,
        confidence=0.99,
        maxIters=10000
    )

    if mask is None:
        print(f"  №1 vs №{target_idx+1}: RANSAC не нашёл модель")
        continue

    mask = mask.ravel().astype(bool)
    inliers = [good[i] for i in range(len(good)) if mask[i]]
    print(f"  №1 vs №{target_idx+1}: {len(good)} сырых -> {len(inliers)} инлайеров (RANSAC)")

    for m in inliers:
        ref_kp_idx = m.queryIdx
        target_kp_idx = m.trainIdx
        if target_idx not in track_dists[ref_kp_idx] or m.distance < track_dists[ref_kp_idx][target_idx]:
            tracks[ref_kp_idx][target_idx] = target_kp_idx
            track_dists[ref_kp_idx][target_idx] = m.distance

voting = {}
for ref_kp_idx, matches_dict in tracks.items():
    total_images = len(matches_dict) + 1
    voting[ref_kp_idx] = total_images

print(f"\nВсего точек-кандидатов (найдены хотя бы на 2 изображениях): {len(tracks)}")

def avg_dist(ref_kp_idx):
    dists = list(track_dists[ref_kp_idx].values())
    return np.mean(dists) if dists else 1e9

sorted_tracks = sorted(
    tracks.items(),
    key=lambda item: (-voting[item[0]], avg_dist(item[0]))
)

TOP_N = 30
top_tracks = sorted_tracks[:TOP_N]

print(f"Отобрано лучших треков: {len(top_tracks)}")
print("Распределение по числу изображений:")
cnt = Counter(len(m) + 1 for _, m in top_tracks)
for k in sorted(cnt.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt[k]} точек")

def draw_tracks_on_image(img, kp_list, track_info, img_idx, is_ref=False):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (ref_kp_idx, matches_dict) in enumerate(track_info, 1):
        if is_ref:
            kp_idx = ref_kp_idx
        else:
            if img_idx not in matches_dict:
                continue
            kp_idx = matches_dict[img_idx]

        x, y = kp_list[kp_idx].pt
        x, y = int(round(x)), int(round(y))

        cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
        cv2.putText(vis, str(track_id), (x - 10, y + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

for i in range(6):
    ax = axes[i]
    if imgs_processed[i] is None:
        ax.axis('off')
        continue

    is_ref = (i == REF_IDX)
    vis_img = draw_tracks_on_image(imgs_processed[i], kps_list[i], top_tracks, i, is_ref=is_ref)
    vis_rgb = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)

    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.imshow(vis_rgb)
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')

plt.suptitle("Общие ключевые точки (AKAZE + RANSAC), найденные на >= 2 изображениях", fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

## Фильтр по среднему расстоянию дескрипторов

In [ ]:
def avg_dist(ref_kp_idx):
    dists = list(track_dists[ref_kp_idx].values())
    return np.mean(dists) if dists else 1e9

valid_tracks = {
    ref_kp_idx: matches_dict
    for ref_kp_idx, matches_dict in tracks.items()
    if avg_dist(ref_kp_idx) < 120
}

print(f"Треков после фильтра по расстоянию: {len(valid_tracks)}")

## Попытка c AKAZE + LOGOS

In [ ]:
akaze = cv2.AKAZE.create(
    threshold=0.00001,
    descriptor_size=0,
    descriptor_channels=3,
    nOctaves=4,
    nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2
)
print("AKAZE инициализирован")

kps_list, descs_list, imgs_processed = [], [], []
for i, im in enumerate(imgs_final):
    if im is None:
        kps_list.append([]); descs_list.append(None); imgs_processed.append(None)
        continue
    if i == 5:
        h, w = im.shape
        scale = 1280 / max(h, w)
        if scale > 1.0:
            im = cv2.resize(im, (int(round(w*scale)), int(round(h*scale))),
                            interpolation=cv2.INTER_CUBIC)
    kp, desc = akaze.detectAndCompute(im, None)
    kps_list.append(kp); descs_list.append(desc); imgs_processed.append(im)
    print(f"№{i+1}: {len(kp)} точек AKAZE")

REF_IDX = 0
tracks = defaultdict(dict)
track_dists = defaultdict(dict)

print("\nСопоставление эталона с остальными (AKAZE + LOGOS):")
for target_idx in range(1, 6):
    desc_ref = descs_list[REF_IDX]
    desc_target = descs_list[target_idx]
    if desc_ref is None or desc_target is None:
        continue

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except cv2.error:
        continue
    good = [m for m, n in matches if m.distance < 0.80 * n.distance]
    if len(good) < 8:
        print(f"  №1 vs №{target_idx+1}: {len(good)} совпадений (мало)")
        continue

    size_ref = (imgs_processed[REF_IDX].shape[1], imgs_processed[REF_IDX].shape[0])
    size_tgt = (imgs_processed[target_idx].shape[1], imgs_processed[target_idx].shape[0])

    try:
        matches_logos = cv2.xfeatures2d.matchLOGOS(
            size_ref, size_tgt,
            kps_list[REF_IDX], kps_list[target_idx],
            good,
            withRotation=True,
            withScale=True
        )
    except Exception as e:
        print(f"  №1 vs №{target_idx+1}: ошибка LOGOS: {e}")
        src_pts = np.float32([kps_list[REF_IDX][m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kps_list[target_idx][m.trainIdx].pt for m in good]).reshape(-1,1,2)
        F, mask = cv2.findFundamentalMat(src_pts, dst_pts, cv2.FM_RANSAC, 6.0, 0.99, 10000)
        matches_logos = [good[i] for i in range(len(good)) if mask is not None and mask[i][0]]

    if matches_logos is None or len(matches_logos) == 0:
        print(f"  №1 vs №{target_idx+1}: LOGOS не нашёл совпадений")
        continue

    print(f"  №1 vs №{target_idx+1}: {len(good)} сырых -> {len(matches_logos)} после LOGOS")

    for m in matches_logos:
        ref_kp_idx = m.queryIdx
        target_kp_idx = m.trainIdx
        if target_idx not in track_dists[ref_kp_idx] or m.distance < track_dists[ref_kp_idx][target_idx]:
            tracks[ref_kp_idx][target_idx] = target_kp_idx
            track_dists[ref_kp_idx][target_idx] = m.distance

voting = {r: len(m) + 1 for r, m in tracks.items()}
def avg_dist(ref_kp_idx):
    dists = list(track_dists[ref_kp_idx].values())
    return np.mean(dists) if dists else 1e9

print(f"\nВсего точек-кандидатов: {len(tracks)}")
sorted_tracks = sorted(tracks.items(), key=lambda item: (-voting[item[0]], avg_dist(item[0])))
top_tracks = sorted_tracks[:30]

print(f"Отобрано лучших треков: {len(top_tracks)}")
cnt = Counter(len(m) + 1 for _, m in top_tracks)
print("Распределение по числу изображений:")
for k in sorted(cnt.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt[k]} точек")

def draw_tracks_on_image(img, kp_list, track_info, img_idx, is_ref=False):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (ref_kp_idx, matches_dict) in enumerate(track_info, 1):
        if is_ref:
            kp_idx = ref_kp_idx
        else:
            if img_idx not in matches_dict:
                continue
            kp_idx = matches_dict[img_idx]
        x, y = kp_list[kp_idx].pt
        x, y = int(round(x)), int(round(y))
        cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
        cv2.putText(vis, str(track_id), (x - 10, y + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()
for i in range(6):
    ax = axes[i]
    if imgs_processed[i] is None:
        ax.axis('off'); continue
    is_ref = (i == REF_IDX)
    vis = draw_tracks_on_image(imgs_processed[i], kps_list[i], top_tracks, i, is_ref=is_ref)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    ax.imshow(vis_rgb)
    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')
plt.suptitle("ПОПЫТКА 3: AKAZE + LOGOS", fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

## Попытка c AKAZE + LOGOS/GMS + RANSAC + маска

In [ ]:
def segment_organism(gray):
    med = float(np.median(gray))
    diff = np.abs(gray.astype(np.float32) - med)
    diff_n = np.clip(diff / (diff.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(diff_n, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open,  iterations=1)
    n_lab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_lab > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest = 1 + int(np.argmax(areas))
        mask = (labels == largest).astype(np.uint8) * 255
    k_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_fill, iterations=3)
    return mask

def is_inside_mask(kp, mask, margin=5):
    x, y = int(round(kp.pt[0])), int(round(kp.pt[1]))
    if 0 <= y < mask.shape[0] and 0 <= x < mask.shape[1]:
        y0, y1 = max(0, y-margin), min(mask.shape[0], y+margin+1)
        x0, x1 = max(0, x-margin), min(mask.shape[1], x+margin+1)
        return mask[y0:y1, x0:x1].mean() > 128
    return False

akaze = cv2.AKAZE.create(
    threshold=0.00001,
    descriptor_size=0,
    descriptor_channels=3,
    nOctaves=4,
    nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2
)
print("AKAZE инициализирован")

kps_list, descs_list, imgs_processed = [], [], []
for i, im in enumerate(imgs_final):
    if im is None:
        kps_list.append([]); descs_list.append(None); imgs_processed.append(None)
        continue
    if i == 5:
        h, w = im.shape
        scale = 1280 / max(h, w)
        if scale > 1.0:
            im = cv2.resize(im, (int(round(w*scale)), int(round(h*scale))),
                            interpolation=cv2.INTER_CUBIC)
    kp, desc = akaze.detectAndCompute(im, None)
    kps_list.append(kp); descs_list.append(desc); imgs_processed.append(im)
    print(f"№{i+1}: {len(kp)} точек AKAZE")

masks_processed = [segment_organism(im) if im is not None else None for im in imgs_processed]
print("\nМаски построены")

REF_IDX = 0
tracks = defaultdict(dict)
track_dists = defaultdict(dict)

print("\nСопоставление (AKAZE + LOGOS + RANSAC + маска):")
for target_idx in range(1, 6):
    desc_ref = descs_list[REF_IDX]
    desc_target = descs_list[target_idx]
    if desc_ref is None or desc_target is None:
        continue

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except cv2.error:
        continue
    good = [m for m, n in matches if m.distance < 0.80 * n.distance]
    if len(good) < 8:
        print(f"  №1 vs №{target_idx+1}: {len(good)} совпадений (мало)")
        continue

    size_ref = (imgs_processed[REF_IDX].shape[1], imgs_processed[REF_IDX].shape[0])
    size_tgt = (imgs_processed[target_idx].shape[1], imgs_processed[target_idx].shape[0])

    try:
        if hasattr(cv2.xfeatures2d, 'matchLOGOS'):
            matches_filtered = cv2.xfeatures2d.matchLOGOS(
                size_ref, size_tgt, kps_list[REF_IDX], kps_list[target_idx],
                good, withRotation=True, withScale=True
            )
        elif hasattr(cv2.xfeatures2d, 'matchGMS'):
            matches_filtered = cv2.xfeatures2d.matchGMS(
                size_ref, size_tgt, kps_list[REF_IDX], kps_list[target_idx],
                good, withRotation=True, withScale=True, thresholdFactor=6.0
            )
        else:
            raise AttributeError("Ни GMS, ни LOGOS не доступны")
    except Exception as e:
        print(f"  №1 vs №{target_idx+1}: пространственный фильтр не сработал ({e}), используем RANSAC")
        matches_filtered = good

    if matches_filtered is None or len(matches_filtered) < 8:
        print(f"  №1 vs №{target_idx+1}: после пространственного фильтра {0 if matches_filtered is None else len(matches_filtered)} (мало)")
        continue

    src_pts = np.float32([kps_list[REF_IDX][m.queryIdx].pt for m in matches_filtered]).reshape(-1, 1, 2)
    dst_pts = np.float32([kps_list[target_idx][m.trainIdx].pt for m in matches_filtered]).reshape(-1, 1, 2)

    F, mask_ransac = cv2.findFundamentalMat(
        src_pts, dst_pts, cv2.FM_RANSAC, 6.0, 0.99, 10000
    )
    if mask_ransac is None:
        inliers = matches_filtered
    else:
        mask_ransac = mask_ransac.ravel().astype(bool)
        inliers = [matches_filtered[i] for i in range(len(matches_filtered)) if mask_ransac[i]]

    filtered = []
    for m in inliers:
        kp_ref = kps_list[REF_IDX][m.queryIdx]
        kp_tgt = kps_list[target_idx][m.trainIdx]
        if is_inside_mask(kp_ref, masks_processed[REF_IDX]) and \
           is_inside_mask(kp_tgt, masks_processed[target_idx]):
            filtered.append(m)

    print(f"  №1 vs №{target_idx+1}: {len(good)} -> {len(matches_filtered)} (пространств.) -> "
          f"{len(inliers)} (RANSAC) -> {len(filtered)} (маска)")

    for m in filtered:
        ref_kp_idx = m.queryIdx
        target_kp_idx = m.trainIdx
        if target_idx not in track_dists[ref_kp_idx] or m.distance < track_dists[ref_kp_idx][target_idx]:
            tracks[ref_kp_idx][target_idx] = target_kp_idx
            track_dists[ref_kp_idx][target_idx] = m.distance

voting = {r: len(m) + 1 for r, m in tracks.items()}
def avg_dist(ref_kp_idx):
    dists = list(track_dists[ref_kp_idx].values())
    return np.mean(dists) if dists else 1e9

print(f"\nВсего треков-кандидатов: {len(tracks)}")
sorted_tracks = sorted(tracks.items(), key=lambda item: (-voting[item[0]], avg_dist(item[0])))
top_tracks = sorted_tracks[:30]

print(f"Отобрано лучших треков: {len(top_tracks)}")
cnt = Counter(len(m) + 1 for _, m in top_tracks)
print("Распределение по числу изображений:")
for k in sorted(cnt.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt[k]} точек")

def draw_tracks_on_image(img, kp_list, track_info, img_idx, is_ref=False):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (ref_kp_idx, matches_dict) in enumerate(track_info, 1):
        if is_ref:
            kp_idx = ref_kp_idx
        else:
            if img_idx not in matches_dict:
                continue
            kp_idx = matches_dict[img_idx]
        x, y = kp_list[kp_idx].pt
        x, y = int(round(x)), int(round(y))
        cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
        cv2.putText(vis, str(track_id), (x - 10, y + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()
for i in range(6):
    ax = axes[i]
    if imgs_processed[i] is None:
        ax.axis('off'); continue
    is_ref = (i == REF_IDX)
    vis = draw_tracks_on_image(imgs_processed[i], kps_list[i], top_tracks, i, is_ref=is_ref)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    ax.imshow(vis_rgb)
    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')
plt.suptitle("ПОПЫТКА 4: AKAZE + LOGOS/GMS + RANSAC + маска", fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

## Попытка с оптическим потоком (Lucas-Kanade)

In [ ]:
MAX_SIDE = 1000

def prepare_images(imgs, max_side=MAX_SIDE):
    processed = []
    sizes = []

    for im in imgs:
        if im is None:
            processed.append(None)
            sizes.append((0, 0))
            continue
        h, w = im.shape[:2]
        scale = max_side / max(h, w)
        new_w = int(round(w * scale))
        new_h = int(round(h * scale))
        resized = cv2.resize(im, (new_w, new_h), interpolation=cv2.INTER_AREA)
        processed.append(resized)
        sizes.append((new_h, new_w))

    max_h = max(s[0] for s in sizes if s[0] > 0)
    max_w = max(s[1] for s in sizes if s[1] > 0)

    padded = []
    for im in processed:
        if im is None:
            padded.append(None)
            continue
        h, w = im.shape[:2]
        border_pixels = np.concatenate([
            im[0, :], im[-1, :], im[:, 0], im[:, -1]
        ])
        bg_value = int(np.median(border_pixels))

        top = (max_h - h) // 2
        bottom = max_h - h - top
        left = (max_w - w) // 2
        right = max_w - w - left

        padded_img = cv2.copyMakeBorder(
            im, top, bottom, left, right,
            cv2.BORDER_CONSTANT, value=bg_value
        )
        padded.append(padded_img)

    return padded, (max_h, max_w)

imgs_uniform, common_size = prepare_images(imgs_final)
print(f"Общий размер изображений: {common_size[1]}×{common_size[0]} (W×H)")
for i, im in enumerate(imgs_uniform):
    if im is not None:
        print(f"  №{i+1}: {im.shape[1]}×{im.shape[0]}")

akaze = cv2.AKAZE.create(
    threshold=0.00001,
    descriptor_size=0,
    descriptor_channels=3,
    nOctaves=4,
    nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2
)

REF_IDX = 0
kp_ref = akaze.detect(imgs_uniform[REF_IDX], None)
print(f"\nНа эталоне (№1) найдено {len(kp_ref)} точек AKAZE")

pts_ref = np.float32([kp.pt for kp in kp_ref]).reshape(-1, 1, 2)

def track_points(img_ref, pts_ref, img_target, win_size=(21, 21), max_level=3):
    lk_params = dict(
        winSize=win_size,
        maxLevel=max_level,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
    )
    pts_target, status, err = cv2.calcOpticalFlowPyrLK(
        img_ref, img_target, pts_ref, None, **lk_params
    )
    return pts_target, status.ravel()

print("\nПрямое отслеживание (№1 -> каждое изображение):")
tracks = {}
for target_idx in range(1, 6):
    if imgs_uniform[target_idx] is None:
        continue
    pts_target, status = track_points(
        imgs_uniform[REF_IDX], pts_ref, imgs_uniform[target_idx]
    )
    success = int(status.sum())
    print(f"  №1 -> №{target_idx+1}: {success} из {len(pts_ref)} точек отслежено")

    for i, (pt, st) in enumerate(zip(pts_target, status)):
        if st == 1:
            if i not in tracks:
                tracks[i] = {REF_IDX: tuple(pts_ref[i].ravel())}
            tracks[i][target_idx] = tuple(pt.ravel())

print("\nПоследовательное отслеживание (цепочка):")
order = [0, 1, 2, 3, 4, 5]

pts_current = pts_ref.copy()
current_img = imgs_uniform[order[0]]
tracks_chain = {i: {order[0]: tuple(pts_ref[i].ravel())} for i in range(len(pts_ref))}

for step in range(1, len(order)):
    next_idx = order[step]
    next_img = imgs_uniform[next_idx]
    if next_img is None or current_img is None:
        continue

    pts_next, status = track_points(current_img, pts_current, next_img)
    success = int(status.sum())
    print(f"  Шаг {step}: №{order[step-1]+1} -> №{next_idx+1}: {success} точек отслежено")

    new_tracks = {}
    for i, (pt, st) in enumerate(zip(pts_next, status)):
        if st == 1 and i in tracks_chain:
            tracks_chain[i][next_idx] = tuple(pt.ravel())
            new_tracks[i] = tracks_chain[i]

    tracks_chain = new_tracks
    if len(tracks_chain) == 0:
        print("  Цепочка оборвалась.")
        break

    valid_keys = list(tracks_chain.keys())
    pts_current = np.float32([tracks_chain[k][next_idx] for k in valid_keys]).reshape(-1, 1, 2)
    tracks_chain = {new_i: tracks_chain[old_k]
                    for new_i, old_k in enumerate(valid_keys)}
    current_img = next_img

print(f"\nДлина цепочки: {len(tracks_chain)} треков после всех шагов")

all_tracks = {i: dict(tr) for i, tr in tracks.items()}
for i, tr in tracks_chain.items():
    if REF_IDX in tr:
        ref_coord = tr[REF_IDX]
        for orig_i, orig_tr in tracks.items():
            if REF_IDX in orig_tr:
                if np.linalg.norm(np.array(orig_tr[REF_IDX]) - np.array(ref_coord)) < 1.0:
                    for img_idx, pt in tr.items():
                        all_tracks[orig_i][img_idx] = pt
                    break

valid_tracks = {i: tr for i, tr in all_tracks.items() if len(tr) >= 3}
print(f"\nТреков, найденных на >= 3 изображениях: {len(valid_tracks)}")

print("Распределение треков по числу изображений:")
cnt_all = Counter(len(tr) for tr in all_tracks.values())
for k in sorted(cnt_all.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt_all[k]} точек")

sorted_tracks = sorted(valid_tracks.items(), key=lambda x: -len(x[1]))
top_tracks = sorted_tracks[:30]

print(f"\nОтобрано лучших треков: {len(top_tracks)}")
cnt = Counter(len(tr) for _, tr in top_tracks)
print("Распределение топ-30 по числу изображений:")
for k in sorted(cnt.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt[k]} точек")

def draw_tracks_on_image(img, track_info, img_idx):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (orig_id, tr) in enumerate(track_info, 1):
        if img_idx in tr:
            x, y = tr[img_idx]
            x, y = int(round(x)), int(round(y))
            cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
            cv2.putText(vis, str(track_id), (x - 10, y + 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()
for i in range(6):
    ax = axes[i]
    if imgs_uniform[i] is None:
        ax.axis('off')
        continue
    is_ref = (i == REF_IDX)
    vis = draw_tracks_on_image(imgs_uniform[i], top_tracks, i)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    ax.imshow(vis_rgb)
    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')

plt.suptitle("ФИНАЛЬНАЯ ПОПЫТКА: Оптический поток (Lucas-Kanade) с общим размером изображений",
             fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("ИТОГОВАЯ СТАТИСТИКА ДЛЯ ОТЧЁТА:")
print("="*60)
print(f"1. Точек на эталоне (№1): {len(kp_ref)}")
print(f"2. Треков найдено (на любом числе изображений): {len(all_tracks)}")
print(f"3. Треков на >= 3 изображениях: {len(valid_tracks)}")
print(f"4. Лучших треков отобрано: {len(top_tracks)}")
if top_tracks:
    print(f"5. Максимальная длина трека: {max(len(tr) for _, tr in top_tracks)} изображений")
print("="*60)

## Финальная попытка: AKAZE с маской на входе детектора

In [ ]:
def segment_organism(gray):
    med = float(np.median(gray))
    diff = np.abs(gray.astype(np.float32) - med)
    diff_n = np.clip(diff / (diff.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(diff_n, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open,  iterations=1)

    n_lab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_lab > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest = 1 + int(np.argmax(areas))
        mask = (labels == largest).astype(np.uint8) * 255

    k_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_fill, iterations=3)

    k_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.erode(mask, k_erode, iterations=1)
    return mask

imgs_processed = []
for i, im in enumerate(imgs_final):
    if im is None:
        imgs_processed.append(None)
        continue
    if i == 5:
        h, w = im.shape
        scale = 1280 / max(h, w)
        if scale > 1.0:
            im = cv2.resize(im, (int(round(w*scale)), int(round(h*scale))),
                            interpolation=cv2.INTER_CUBIC)
    imgs_processed.append(im)

masks = [segment_organism(im) if im is not None else None for im in imgs_processed]
print("Покрытие масок (% кадра):")
for i, m in enumerate(masks):
    if m is not None:
        print(f"  №{i+1}: {(m > 0).mean() * 100:.1f}%")

akaze = cv2.AKAZE.create(
    threshold=0.00001,
    descriptor_size=0,
    descriptor_channels=3,
    nOctaves=4,
    nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2
)

kps_list, descs_list = [], []
print("\nТочки AKAZE ВНУТРИ маски организма:")
for i, im in enumerate(imgs_processed):
    if im is None:
        kps_list.append([]); descs_list.append(None); continue
    kp, desc = akaze.detectAndCompute(im, masks[i])
    kps_list.append(kp); descs_list.append(desc)
    print(f"  №{i+1}: {len(kp)} точек")

REF_IDX = 0
tracks = defaultdict(dict)
track_dists = defaultdict(dict)

print("\nСопоставление (AKAZE + RANSAC):")
for target_idx in range(1, 6):
    desc_ref = descs_list[REF_IDX]
    desc_target = descs_list[target_idx]
    if desc_ref is None or desc_target is None:
        continue

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except cv2.error:
        continue
    good = [m for m, n in matches if m.distance < 0.80 * n.distance]
    if len(good) < 8:
        print(f"  №1 vs №{target_idx+1}: {len(good)} совпадений (мало)")
        continue

    src_pts = np.float32([kps_list[REF_IDX][m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kps_list[target_idx][m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    F, mask_r = cv2.findFundamentalMat(src_pts, dst_pts, cv2.FM_RANSAC, 6.0, 0.99, 10000)

    if mask_r is None:
        print(f"  №1 vs №{target_idx+1}: RANSAC не нашёл модель")
        continue
    mask_r = mask_r.ravel().astype(bool)
    inliers = [good[i] for i in range(len(good)) if mask_r[i]]
    print(f"  №1 vs №{target_idx+1}: {len(good)} сырых -> {len(inliers)} инлайеров")

    for m in inliers:
        ref_kp_idx = m.queryIdx
        target_kp_idx = m.trainIdx
        if target_idx not in track_dists[ref_kp_idx] or m.distance < track_dists[ref_kp_idx][target_idx]:
            tracks[ref_kp_idx][target_idx] = target_kp_idx
            track_dists[ref_kp_idx][target_idx] = m.distance

def avg_dist(ref_kp_idx):
    dists = list(track_dists[ref_kp_idx].values())
    return np.mean(dists) if dists else 1e9

valid_tracks = {
    r: m for r, m in tracks.items()
    if avg_dist(r) < 80
}
print(f"\nТреков после фильтра дескрипторов (avg_dist < 80): {len(valid_tracks)}")

sorted_tracks = sorted(
    valid_tracks.items(),
    key=lambda item: (-len(item[1]), avg_dist(item[0]))
)
top_tracks = sorted_tracks[:30]

print(f"Отобрано лучших треков: {len(top_tracks)}")
cnt = Counter(len(m) + 1 for _, m in top_tracks)
print("Распределение по числу изображений:")
for k in sorted(cnt.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt[k]} точек")

def draw_tracks_on_image(img, kp_list, track_info, img_idx, is_ref=False):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (ref_kp_idx, matches_dict) in enumerate(track_info, 1):
        if is_ref:
            kp_idx = ref_kp_idx
        else:
            if img_idx not in matches_dict:
                continue
            kp_idx = matches_dict[img_idx]
        x, y = kp_list[kp_idx].pt
        x, y = int(round(x)), int(round(y))
        cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
        cv2.putText(vis, str(track_id), (x - 10, y + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()
for i in range(6):
    ax = axes[i]
    if imgs_processed[i] is None:
        ax.axis('off'); continue
    is_ref = (i == REF_IDX)
    vis = draw_tracks_on_image(imgs_processed[i], kps_list[i], top_tracks, i, is_ref=is_ref)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    ax.imshow(vis_rgb)
    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')
plt.suptitle("AKAZE с маской на входе детектора + RANSAC + фильтр дескрипторов",
             fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## Ещё попытки что-то сделать

### LMedS + голосование

In [ ]:
sift_tuned = cv2.SIFT_create(contrastThreshold=0.01, edgeThreshold=20, nOctaveLayers=5)
sift_soft  = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=15)

imgs_processed = []
kps_list, descs_list = [], []

for i in range(6):
    im = imgs_final[i].copy()
    if i == 5:
        h, w = im.shape
        scale = 1280 / max(h, w)
        if scale > 1.0:
            im = cv2.resize(im, (int(round(w*scale)), int(round(h*scale))),
                            interpolation=cv2.INTER_CUBIC)
        detector = sift_soft
    else:
        detector = sift_tuned

    kp, desc = detector.detectAndCompute(im, None)
    imgs_processed.append(im)
    kps_list.append(kp)
    descs_list.append(desc)
    print(f"№{i+1}: {len(kp)} точек")

REF_IDX = 0
tracks = defaultdict(dict)
track_scores = defaultdict(float)

print("\nСопоставление эталона с остальными (SIFT + LMedS):")
for target_idx in range(1, 6):
    desc_ref = descs_list[REF_IDX]
    desc_target = descs_list[target_idx]
    if desc_ref is None or desc_target is None:
        continue

    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except cv2.error:
        continue

    good = [m for m, n in matches if m.distance < 0.8 * n.distance]
    if len(good) < 8:
        print(f"  №1 vs №{target_idx+1}: {len(good)} совпадений (мало для LMedS)")
        continue

    src_pts = np.float32([kps_list[REF_IDX][m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kps_list[target_idx][m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    F, mask = cv2.findFundamentalMat(
        src_pts, dst_pts,
        method=cv2.FM_LMEDS,
        ransacReprojThreshold=3.0,
        confidence=0.99,
        maxIters=5000
    )

    if mask is None:
        print(f"  №1 vs №{target_idx+1}: LMedS не нашёл модель")
        continue

    mask = mask.ravel().astype(bool)
    inliers = [good[i] for i in range(len(good)) if mask[i]]
    print(f"  №1 vs №{target_idx+1}: {len(good)} сырых -> {len(inliers)} инлайеров (LMedS)")

    for m in inliers:
        ref_kp_idx = m.queryIdx
        target_kp_idx = m.trainIdx
        if target_idx not in tracks[ref_kp_idx] or m.distance < track_scores[ref_kp_idx]:
            tracks[ref_kp_idx][target_idx] = target_kp_idx
            track_scores[ref_kp_idx] = m.distance

voting = {}
for ref_kp_idx, matches_dict in tracks.items():
    total_images = len(matches_dict) + 1
    voting[ref_kp_idx] = total_images

sorted_tracks = sorted(
    tracks.items(),
    key=lambda item: (-voting[item[0]], track_scores[item[0]])
)

TOP_N = 30
top_tracks = sorted_tracks[:TOP_N]

print(f"\nВсего точек-кандидатов (найдены хотя бы на 2 изображениях): {len(tracks)}")
print(f"Отобрано лучших треков: {len(top_tracks)}")
print(f"Распределение по числу изображений:")
for k in sorted(set(voting.values()), reverse=True):
    count = sum(1 for v in voting.values() if v == k)
    print(f"  найдено на {k} изображениях: {count} точек")

def draw_tracks_on_image(img, kp_list, track_info, img_idx, is_ref=False):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (ref_kp_idx, matches_dict) in enumerate(track_info, 1):
        if is_ref:
            kp_idx = ref_kp_idx
        else:
            if img_idx not in matches_dict:
                continue
            kp_idx = matches_dict[img_idx]
        x, y = kp_list[kp_idx].pt
        x, y = int(round(x)), int(round(y))
        cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
        cv2.putText(vis, str(track_id), (x - 10, y + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

for i in range(6):
    ax = axes[i]
    if imgs_processed[i] is None:
        ax.axis('off')
        continue
    is_ref = (i == REF_IDX)
    vis_img = draw_tracks_on_image(imgs_processed[i], kps_list[i], top_tracks, i, is_ref=is_ref)
    vis_rgb = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)
    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.imshow(vis_rgb)
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')

plt.suptitle("Общие ключевые точки (SIFT + LMedS), найденные на >= 2 изображениях", fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

### Фильтрация треков по маске и консистентности дескрипторов

In [ ]:
sift_tuned = cv2.SIFT_create(contrastThreshold=0.01, edgeThreshold=20, nOctaveLayers=5)
sift_soft  = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=15)

imgs_processed = []
kps_list, descs_list = [], []

for i in range(6):
    im = imgs_final[i].copy()
    if i == 5:
        h, w = im.shape
        scale = 1280 / max(h, w)
        if scale > 1.0:
            im = cv2.resize(im, (int(round(w*scale)), int(round(h*scale))),
                            interpolation=cv2.INTER_CUBIC)
        detector = sift_soft
    else:
        detector = sift_tuned

    kp, desc = detector.detectAndCompute(im, None)
    imgs_processed.append(im)
    kps_list.append(kp)
    descs_list.append(desc)
    print(f"№{i+1}: {len(kp)} точек")

tracks = defaultdict(dict)
track_dists = defaultdict(dict)

for target_idx in range(1, 6):
    desc_ref = descs_list[REF_IDX]
    desc_target = descs_list[target_idx]
    if desc_ref is None or desc_target is None:
        continue
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    try:
        matches = bf.knnMatch(desc_ref, desc_target, k=2)
    except:
        continue
    good = [m for m, n in matches if m.distance < 0.8 * n.distance]
    if len(good) < 8:
        continue
    src_pts = np.float32([kps_list[REF_IDX][m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst_pts = np.float32([kps_list[target_idx][m.trainIdx].pt for m in good]).reshape(-1,1,2)
    F, mask = cv2.findFundamentalMat(src_pts, dst_pts, cv2.FM_LMEDS, 3.0, 0.99, 5000)
    if mask is None:
        continue
    mask = mask.ravel().astype(bool)
    for i, m in enumerate(good):
        if mask[i]:
            ref_kp_idx = m.queryIdx
            target_kp_idx = m.trainIdx
            if target_idx not in track_dists[ref_kp_idx] or m.distance < track_dists[ref_kp_idx][target_idx]:
                tracks[ref_kp_idx][target_idx] = target_kp_idx
                track_dists[ref_kp_idx][target_idx] = m.distance

def is_in_mask(kp, mask, img_idx):
    x, y = int(round(kp.pt[0])), int(round(kp.pt[1]))
    if 0 <= y < mask.shape[0] and 0 <= x < mask.shape[1]:
        return mask[y, x] > 0
    return False

def segment_organism(gray):
    med = float(np.median(gray))
    diff = np.abs(gray.astype(np.float32) - med)
    diff_n = np.clip(diff / (diff.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(diff_n, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k_open, iterations=1)
    n_lab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_lab > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest = 1 + int(np.argmax(areas))
        mask = (labels == largest).astype(np.uint8) * 255
    k_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_fill, iterations=3)
    return mask

masks_processed = [segment_organism(im) for im in imgs_processed]

track_avg_dist = {}
for ref_kp_idx, matches_dict in tracks.items():
    dists = list(track_dists[ref_kp_idx].values())
    if len(dists) >= 1:
        track_avg_dist[ref_kp_idx] = np.mean(dists)
    else:
        track_avg_dist[ref_kp_idx] = 1e9

valid_tracks = {}
for ref_kp_idx, matches_dict in tracks.items():
    total_images = len(matches_dict) + 1
    if total_images >= 2 and track_avg_dist[ref_kp_idx] < 150:
        ok = True
        if not is_in_mask(kps_list[REF_IDX][ref_kp_idx], masks_processed[REF_IDX], REF_IDX):
            ok = False
        if ok:
            for target_idx, target_kp_idx in matches_dict.items():
                if not is_in_mask(kps_list[target_idx][target_kp_idx], masks_processed[target_idx], target_idx):
                    ok = False
                    break
        if ok:
            valid_tracks[ref_kp_idx] = matches_dict

print(f"\nТреков после фильтрации (маска + расстояние < 150): {len(valid_tracks)}")

sorted_tracks = sorted(
    valid_tracks.items(),
    key=lambda item: (- (len(item[1]) + 1), track_avg_dist[item[0]])
)
TOP_N = 30
top_tracks = sorted_tracks[:TOP_N]

print(f"Отобрано лучших треков: {len(top_tracks)}")
print("Распределение по числу изображений:")
cnt = Counter(len(m)+1 for _, m in top_tracks)
for k in sorted(cnt.keys(), reverse=True):
    print(f"  на {k} изображениях: {cnt[k]} точек")

def draw_tracks_on_image(img, kp_list, track_info, img_idx, is_ref=False):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for track_id, (ref_kp_idx, matches_dict) in enumerate(track_info, 1):
        if is_ref:
            kp_idx = ref_kp_idx
        else:
            if img_idx not in matches_dict:
                continue
            kp_idx = matches_dict[img_idx]
        x, y = kp_list[kp_idx].pt
        x, y = int(round(x)), int(round(y))
        cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
        cv2.putText(vis, str(track_id), (x - 10, y + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return vis

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()
for i in range(6):
    ax = axes[i]
    if imgs_processed[i] is None:
        ax.axis('off')
        continue
    is_ref = (i == REF_IDX)
    vis_img = draw_tracks_on_image(imgs_processed[i], kps_list[i], top_tracks, i, is_ref=is_ref)
    vis_rgb = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)
    title = f"№{i+1}" + (" (ЭТАЛОН)" if is_ref else "")
    ax.imshow(vis_rgb)
    ax.set_title(title, fontsize=14, fontweight='bold' if is_ref else 'normal')
    ax.axis('off')
plt.suptitle("Общие ключевые точки (SIFT + LMedS + маска + фильтр дескрипторов)", fontsize=18, y=0.98)
plt.tight_layout()
plt.show()